# Bayesian inference: CIP dose-response model

Runs every fit and writes all intermediate results (NetCDF posteriors and CSV tables) used by `main_figures.ipynb` and `supp_figures.ipynb`. **Run this notebook first.**

**Re-running.** Every fit checks whether its output file already exists and, if so, loads it instead of sampling. Running the notebook top to bottom therefore rebuilds every CSV table from the stored posteriors in a few minutes. To redo a fit, delete its `.nc` file (or pass `refit=True`). Sampling is seeded (`RANDOM_SEED` in `functions.py`).

| Section | Output | Used in |
|---|---|---|
| 1. Model fits, standard medium | `results/cip_model_standard_{rep}_12h.nc` | Figs. 1, 2, S2–S5, S6 |
| 2. Independent logistic fits | `results/cip_logistic_standard_{rep}_dose_{c}_12h.nc` | Figs. 1B, S2–S5 B |
| 3. r(c) and K(c) | `results/rK_{rep}_12h.csv` | Figs. 1C–D, S2–S5 C–D |
| 4. MIC from the *n* lowest doses | `results/mic_calculations_add_{n}_12h.nc`, `results/mic_convergence_data_linear_add.csv` | Fig. 2A |
| 4b. MIC from all dose subsets | `results/mic_calculations_combinations_{id}_12h.nc`, `results/mic_convergence_combinations_12h.csv`, `results/rhat_scores_combinations_12h.csv` | Fig. S6 |
| 5. Coefficients of variation | `results/cip_model_cv.csv` | Fig. 2B |
| 6. Medium perturbations | `results/cip_model_{medium}_{cond}_12h.nc`, `results/parameters_table.csv`, `results/{TMP,CHL}_linear_fit.nc` | Table 1, Figs. 3–5, S1 |
| 7. Fixed-α fits | `results/cip_model_{medium}_{cond}_12h_fixed_alpha.nc`, `results/parameters_table_fixed_alpha.csv`, `results/{TMP,CHL}_linear_fit_fixed_alpha.nc` | Fig. S7 |

CIP concentrations are in µg/mL throughout (figures show ng/mL).

In [1]:
from functions import *

os.makedirs('results', exist_ok=True)

---
## 1. Model fits — standard medium
*(Figs. 1, 2, S2–S5)*

Equation (14) is fitted **simultaneously** to all 11 CIP concentrations of each of the five biological replicates (M9 + 0.2% glucose, $N_0 \approx 0.001$), with one shared parameter set $(r_0, K_0, q, c_0, \alpha)$. Only the first 12 h of each curve are used.

The diagnostics table should show $\hat R < 1.01$ and no divergences for every replicate; if a chain gets stuck, delete that file and refit.

In [2]:
DF_std = load_data('standard')

diag = {}
for rep in range(1, 6):
    idata = fit_model(DF_std[DF_std['rep'] == rep],
                      f'results/cip_model_standard_{rep}_12h.nc', Y0_STANDARD)
    diag[rep] = diagnostics(idata)

pd.DataFrame(diag).T

Initializing NUTS using jitter+adapt_full...
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pytensor/link/c/cmodule.py:3004: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature

Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 182 seconds.
Initializing NUTS using jitter+adapt_full...
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature")
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 140 seconds.
Initializing NUTS using jitter+adapt_full...
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature")
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

... (2 outputs left out)

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 118 seconds.
Initializing NUTS using jitter+adapt_full...
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature")
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 129 seconds.


,max_rhat,min_ess,n_divergent
1,1.000906,2596.274283,0.0
2,1.002912,4741.131032,0.0
3,1.001594,4890.941750,0.0
4,1.002337,5829.833803,0.0
5,1.000639,5511.100322,0.0


---
## 2. Independent logistic fits (comparison model)
*(Figs. 1B, S2–S5 panel B)*

A plain logistic $\dot N = rN(1 - N/K)$ fitted **separately** to each (replicate, concentration) pair, with no information shared across concentrations. Near and above the MIC the nearly flat curves make $(r, K)$ unidentifiable.

In [3]:
for rep in range(1, 6):
    DF_rep = DF_std[DF_std['rep'] == rep]
    for c in sorted(DF_rep['dosisAb'].unique()):
        fit_logistic(DF_rep[DF_rep['dosisAb'] == c], logistic_filename(rep, c), Y0_STANDARD)

---
## 3. r(c) and K(c) from the posteriors
*(Figs. 1C–D, S2–S5 panels C–D)*

For each replicate, $r(c)$ and $K(c)$ (Eq. 13) are evaluated at every CIP concentration for 1000 random posterior samples and summarised as mean ± SD. Samples above the MIC are set to $r = K = 0$. For the independent logistic fits, $r$ and $K$ are direct MCMC parameters.

In [4]:
doses = sorted(DF_std['dosisAb'].unique())

for rep in range(1, 6):
    idata = az.from_netcdf(f'results/cip_model_standard_{rep}_12h.nc')
    Ravg, Rstd, Kavg, Kstd = rK_summary(idata, doses)

    RavgL, RstdL, KavgL, KstdL = [], [], [], []
    for c in doses:
        p = posterior_draws(az.from_netcdf(logistic_filename(rep, c)), 1000, ['r', 'K'])
        RavgL.append(p['r'].mean()); RstdL.append(p['r'].std())
        KavgL.append(p['K'].mean()); KstdL.append(p['K'].std())

    pd.DataFrame({'c': doses,
                  'Ravg': Ravg, 'Rstd': Rstd, 'Kavg': Kavg, 'Kstd': Kstd,
                  'RavgL': RavgL, 'RstdL': RstdL, 'KavgL': KavgL, 'KstdL': KstdL}
                 ).to_csv(f'results/rK_{rep}_12h.csv', index=False)

---
## 4. MIC from the *n* lowest concentrations
*(Fig. 2A)*

How many concentrations are needed for a reliable MIC? Using replicate 2, the model is refitted to the $n$ lowest concentrations ($n = 3, \dots, 11$, always including the drug-free control) and the full posterior MIC distribution is stored for each $n$.

In [ ]:
REF_REP = 2   # replicate shown in Fig. 1
DF_ref = DF_std[DF_std['rep'] == REF_REP]
CIP_conc = sorted(DF_ref['dosisAb'].unique())

mic_ref, mic_ref_sd = get_mic(az.from_netcdf(f'results/cip_model_standard_{REF_REP}_12h.nc'))
print(f'Reference MIC (all 11 concentrations): {1000 * mic_ref:.2f} +/- {1000 * mic_ref_sd:.2f} ng/mL')

rows = []
for n in range(3, len(CIP_conc) + 1):
    subset = CIP_conc[:n]
    idata = fit_model(DF_ref[DF_ref['dosisAb'].isin(subset)],
                      f'results/mic_calculations_add_{n}_12h.nc', Y0_STANDARD)
    rows.append(pd.DataFrame({'num_conc': n, 'mic_value': get_mic(idata, 'full')}))

pd.concat(rows).to_csv('results/mic_convergence_data_linear_add.csv', index=False)

Reference MIC (all 11 concentrations): 10.77 +/- 0.10 ng/mL


Initializing NUTS using jitter+adapt_full...
/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature")
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

### 4b. MIC from every subset of the sub-MIC concentrations
*(Fig. S6)*

The control (0 ng/mL) is always included; each of the next six concentrations (3–18 ng/mL) is either in or out, giving the subsets encoded by a 6-bit `binary_id` (bit *j* = 1 means the *j*-th of those concentrations is used). The empty subset (control only) cannot be fitted, which leaves 63 fits. Convergence diagnostics of every fit are stored too, because Fig. S6 colours each box by its number of divergences.

In [6]:
control, pool = CIP_conc[0], CIP_conc[1:7]

rows, diag_rows = [], []
for i in range(1, 2 ** len(pool)):
    binary_id = format(i, f'0{len(pool)}b')
    subset = [control] + [c for c, bit in zip(pool, binary_id) if bit == '1']
    idata = fit_model(DF_ref[DF_ref['dosisAb'].isin(subset)],
                      f'results/mic_calculations_combinations_{binary_id}_12h.nc', Y0_STANDARD)
    rows.append(pd.DataFrame({'binary_id': binary_id, 'mic_value': get_mic(idata, 'full')}))
    diag_rows.append({'binary_id': binary_id, **diagnostics(idata)})

pd.concat(rows).to_csv('results/mic_convergence_combinations_12h.csv', index=False)
scores = pd.DataFrame(diag_rows).set_index('binary_id')
scores.to_csv('results/rhat_scores_combinations_12h.csv')
scores.sort_values('n_divergent', ascending=False).head(10)

,max_rhat,min_ess,n_divergent
binary_id,,,
001101,1.011827,262.330477,2957
001001,1.011941,208.271247,2863
000101,1.017099,267.086226,2670
010001,1.618588,6.735387,2580
001011,1.019008,222.244941,2530
001111,1.013745,256.579224,2481
000011,1.010717,382.182339,2374
000111,1.029824,271.702926,2090
001010,1.015913,474.391507,1932


---
## 5. MIC robustness: coefficients of variation
*(Fig. 2B)*

The individual parameters (especially $q$) can have wide posteriors while the MIC stays tightly constrained. For each replicate we compute the posterior coefficient of variation, CV = SD / mean × 100%, of every parameter and of the MIC. The sensitivity analysis (Eq. 17) explains why: the sensitivity of $c^*$ to $q$ is damped by $1 + \alpha\,x/(1+x)$, $x = (c^*/c_0)^\alpha$.

In [7]:
def posterior_cv(idata):
    mic, mic_sd = get_mic(idata)
    cv = {'MIC': 100 * mic_sd / mic}
    for par in ['r0', 'q', 'c0', 'alpha']:
        x = idata.posterior[par]
        cv[par] = float(100 * x.std() / x.mean())
    return cv

DFL = pd.DataFrame([posterior_cv(az.from_netcdf(f'results/cip_model_standard_{rep}_12h.nc'))
                    for rep in range(1, 6)])
DFL.to_csv('results/cip_model_cv.csv', index=False)
DFL.round(1)

,MIC,r0,q,c0,alpha
0,1.3,0.3,11.5,2.0,2.4
1,0.9,0.5,8.6,2.3,2.6
2,0.7,0.3,5.3,1.6,2.9
3,0.7,0.4,4.1,2.5,3.4
4,0.7,0.4,5.7,2.1,2.3


---
## 6. Medium perturbations
*(Table 1, Figs. 3–5, S1)*

Four perturbations at three doses each, every condition fitted independently with its own five parameters:

| Medium | Doses | Type |
|---|---|---|
| glucose | 0.2, 0.025, 0.00625 % (w/v) | carbon source |
| glycerol | 0.2, 0.025, 0.00625 % (w/v) | alternative carbon source |
| TMP | 0, 0.4, 1.0 µg/mL | bacteriostatic (folate synthesis) |
| CHL | 0, 1.4, 6.0 µg/mL | bacteriostatic (translation) |

This series was inoculated at $N_0 \approx 0.003$. Rows F and G of the CHL plate are excluded by `load_data()` (plating failure: no growth in any well, including the drug-free controls).

In [15]:
diag = {}
for medium in MEDIA:
    DF_med = load_data(medium)
    for cond in medium_conditions(DF_med):
        idata = fit_model(DF_med[DF_med['dosis'] == cond],
                          f'results/cip_model_{medium}_{cond}_12h.nc', Y0_MEDIA, draws=4000)
        diag[(medium, cond)] = diagnostics(idata)

pd.DataFrame(diag).T

max_rhat      min_ess  n_divergent
glucose  0.20000  1.000740  2955.047293          0.0
         0.02500  1.000891  3208.491898          6.0
         0.00625  1.000379  8222.938172          4.0
glycerol 0.20000  1.000527  5663.192366         34.0
         0.02500  1.000537  8295.097962          7.0
         0.00625  1.000548  9308.258508          7.0
TMP      1.00000  1.548221     6.955802       3768.0
         0.40000  1.000957  8448.733800          4.0
         0.00000  1.000788  5869.540779          2.0
CHL      6.00000  1.000663  8565.404391         18.0
         1.40000  1.000675  5222.090293          9.0
         0.00000  1.004613  1251.809352         98.0

### 6b. Parameter table (Table 1)

Posterior median and 2.5% / 97.5% quantiles of the five parameters and of the MIC, one row per condition.

In [16]:
def parameter_table(suffix=''):
    rows = []
    for medium in MEDIA:
        for cond in medium_conditions(load_data(medium)):
            idata = az.from_netcdf(f'results/cip_model_{medium}_{cond}_12h{suffix}.nc')
            rows.append({'medium': medium, 'concentration': cond, **summarise_posterior(idata)})
    return pd.DataFrame(rows)

ParDF = parameter_table()
ParDF.to_csv('results/parameters_table.csv', index=False)

# Table 1 layout: c0 and MIC in ng/mL
show = ParDF.set_index(['medium', 'concentration'])[[f'{p}_median' for p in MODEL_PARAMS + ['MIC']]].copy()
show[['c0_median', 'MIC_median']] *= 1000
show.round(3)

r0_median  K0_median  q_median  c0_median  \
medium   concentration                                              
glucose  0.20000            1.076      0.350    17.386     15.389   
         0.02500            1.206      0.212    30.424     17.793   
         0.00625            1.146      0.165    40.147     92.838   
glycerol 0.20000            0.930      0.178    24.852     64.761   
         0.02500            0.837      0.173    31.223     85.935   
         0.00625            1.087      0.153    32.329     86.512   
TMP      1.00000            0.539      0.076     7.501     69.951   
         0.40000            0.611      0.246    19.299     98.152   
         0.00000            1.108      0.338    21.075     16.912   
CHL      6.00000            0.512      0.017     0.070     72.897   
         1.40000            0.668      0.226     6.332     21.752   
         0.00000            1.059      0.335    19.423     16.724   

                        alpha_median  MIC_median  
medium   concentration                            
glucose  0.20000               5.951      17.668  
         0.02500               7.549      17.487  
         0.00625               6.827      24.893  
glycerol 0.20000               5.509      37.569  
         0.02500               6.543      31.849  
         0.00625               6.411      30.784  
TMP      1.00000               5.433      90.112  
         0.40000               7.041      51.046  
         0.00000               3.546      18.978  
CHL      6.00000               5.750     168.227  
         1.40000               7.554      26.826  
         0.00000               7.015      18.228

### 6c. q vs r0 across the bacteriostatic conditions
*(Fig. 5A)*

A Bayesian linear regression $q = a + b\,r_0$ through the three TMP and the three CHL conditions (posterior medians), drawn as guides in Fig. 5A, and the Pearson correlations quoted in the Fig. 5 caption.

In [10]:
for medium in ['TMP', 'CHL']:
    df = ParDF[ParDF['medium'] == medium]
    fit_linear(df['r0_median'], df['q_median'], f'results/{medium}_linear_fit.nc')

def pearson(media, table=ParDF):
    df = table[table['medium'].isin(media)]
    return np.corrcoef(df['r0_median'], df['q_median'])[0, 1]

for label, media in [('bacteriostatic (TMP + CHL)', ['TMP', 'CHL']), ('TMP', ['TMP']),
                     ('CHL', ['CHL']), ('carbon source (glucose + glycerol)', ['glucose', 'glycerol'])]:
    print(f'Pearson r(q, r0), {label}: {pearson(media):.2f}')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 6 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 8 seconds.
There were 38 divergences after tuning. Increase `target_accept` or reparameterize.
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Pearson r(q, r0), bacteriostatic (TMP + CHL): 0.76
Pearson r(q, r0), TMP: 0.69
Pearson r(q, r0), CHL: 1.00
Pearson r(q, r0), carbon source (glucose + glycerol): 0.22


---
## 7. Fixed-α fits (sensitivity analysis)
*(Fig. S7)*

The fits of Section 6 repeated with $\alpha \sim \mathcal N(6.0, 0.1^2)$ (`ALPHA_FIXED` in `functions.py`), effectively fixed but still differentiable for NUTS. These fits use PyMC's default `jitter+adapt_diag` initialisation.

In [11]:
diag = {}
for medium in MEDIA:
    DF_med = load_data(medium)
    for cond in medium_conditions(DF_med):
        idata = fit_model(DF_med[DF_med['dosis'] == cond],
                          f'results/cip_model_{medium}_{cond}_12h_fixed_alpha.nc', Y0_MEDIA,
                          alpha_fixed=ALPHA_FIXED, init='jitter+adapt_diag')
        diag[(medium, cond)] = diagnostics(idata)

ParDF_fixed = parameter_table('_fixed_alpha')
ParDF_fixed.to_csv('results/parameters_table_fixed_alpha.csv', index=False)
pd.DataFrame(diag).T

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 177 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 134 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

... (16 outputs left out)

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 42 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [r0, K0, q, c0, alpha, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 36 seconds.


max_rhat      min_ess  n_divergent
glucose  0.20000  1.003151  1577.527139          0.0
         0.02500  1.001052  2202.832723          0.0
         0.00625  1.002054  2496.193687          0.0
glycerol 0.20000  1.003763  1266.050820          0.0
         0.02500  1.003816  2145.175366          0.0
         0.00625  1.001864  2257.048741          0.0
TMP      1.00000  1.002666  2261.736644          0.0
         0.40000  1.002439  2227.885523          0.0
         0.00000  1.001378  2085.933255          0.0
CHL      6.00000  1.004078  1982.389556          0.0
         1.40000  1.001925  1739.144905          0.0
         0.00000  1.002755  1850.759313          0.0

In [12]:
# Linear guides for Fig. S7A (CHL needs a higher target_accept)
for medium, ta in [('TMP', 0.95), ('CHL', 0.99)]:
    df = ParDF_fixed[ParDF_fixed['medium'] == medium]
    idata = fit_linear(df['r0_median'], df['q_median'],
                       f'results/{medium}_linear_fit_fixed_alpha.nc', target_accept=ta)
    a, b = (float(idata.posterior[v].mean()) for v in ('a', 'b'))
    print(f'{medium}: q = {a:.2f} + {b:.2f} r0')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

/home/pcatalan/micromamba/envs/pymc/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 7 seconds.


TMP: q = 0.95 + 21.28 r0


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 18 seconds.
There were 9 divergences after tuning. Increase `target_accept` or reparameterize.
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


CHL: q = -17.97 + 35.04 r0
